# DATASCI 531 – Assignment 3: OpenAlex API

This notebook answers the four questions using the OpenAlex **Works** API. It is structured with clear sections, helper functions, and quality checks.

**How to use:**
1. Set your email in `MAILTO` below (OpenAlex asks you to add a `mailto` parameter for performance and support).
2. Run the cells in order.
3. If you get rate-limit or network errors, re-run the previous cell after a short pause.

References: OpenAlex docs for filtering, searching, grouping, paging.


In [ ]:
# === Setup ===
import requests
import pandas as pd
import time
from datetime import datetime
import matplotlib.pyplot as plt

BASE = "https://api.openalex.org/works"
MAILTO = "[email protected]"  # <- replace with your email for polite requests

pd.set_option("display.max_colwidth", 120)

In [ ]:
# === Helper: GET wrapper with basic retry ===
def oa_get(params, sleep=1.0, max_retries=3):
    params = {k: v for k, v in params.items() if v is not None}
    if MAILTO:
        params["mailto"] = MAILTO
    for attempt in range(max_retries):
        r = requests.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=60)
        if r.status_code == 200:
            return r.json()
        time.sleep(sleep * (attempt + 1))
    r.raise_for_status()


In [ ]:
# === Helpers specific to this assignment ===
def count_title_contains(term):
    """Return total count of works whose *title* contains `term`. Uses title.search.
    Minimizes payload by requesting 1 item per page and reading the meta.count.
    """
    params = {
        "filter": f"title.search:{term}",
        "per-page": 1
    }
    data = oa_get(params)
    return data.get("meta", {}).get("count", None)

def group_by_country_since_2020(term, year_end=None):
    """Group counts by authors' affiliation country_code since 2020 (inclusive).
    Uses group_by=authorships.institutions.country_code.
    """
    if year_end is None:
        year_end = datetime.now().year
    params = {
        "filter": f"title.search:{term},publication_year:2020-{year_end}",
        "group_by": "authorships.institutions.country_code",
        "per-page": 200,  # enough to get all countries
        "sort": "count:desc"
    }
    data = oa_get(params)
    groups = pd.DataFrame(data.get("group_by", []))
    if not groups.empty:
        groups = groups.rename(columns={"key": "country_code", "count": "n"})
    return groups

def top_cited_ai_since_2020(term="artificial intelligence", k=10, year_end=None):
    """Return a dataframe of the top-k most cited works since 2020 with `term` in the title.
    Extracts title, cited_by_count, authors, and institutions.
    """
    if year_end is None:
        year_end = datetime.now().year
    params = {
        "filter": f"title.search:{term},publication_year:2020-{year_end}",
        "sort": "cited_by_count:desc",
        "per-page": k,
        # Keep payload small: choose important fields
        "select": ",".join([
            "id", "display_name", "publication_year", "cited_by_count",
            "authorships.author.display_name",
            "authorships.institutions.display_name"
        ])
    }
    data = oa_get(params)
    works = data.get("results", [])
    rows = []
    for w in works:
        title = w.get("display_name")
        cites = w.get("cited_by_count")
        year = w.get("publication_year")
        # authorships is a list; collect names and institution names
        authors = []
        insts = []
        for auth in w.get("authorships", []):
            a = auth.get("author", {}).get("display_name")
            if a:
                authors.append(a)
            for inst in auth.get("institutions", []):
                name = inst.get("display_name")
                if name:
                    insts.append(name)
        # de-duplicate institutions while preserving order
        seen = set()
        insts_unique = [x for x in insts if not (x in seen or seen.add(x))]
        rows.append({
            "title": title,
            "year": year,
            "cited_by_count": cites,
            "authors": ", ".join(authors),
            "institutions": ", ".join(insts_unique)
        })
    return pd.DataFrame(rows)

def yearly_counts(term, year_start=2000, year_end=None):
    """Return a (year, count) dataframe for titles containing `term` grouped by publication_year."""
    if year_end is None:
        year_end = datetime.now().year
    params = {
        "filter": f"title.search:{term},publication_year:{year_start}-{year_end}",
        "group_by": "publication_year",
        "per-page": 200,
        "sort": "key:asc"
    }
    data = oa_get(params)
    df = pd.DataFrame(data.get("group_by", []))
    if not df.empty:
        df = df.rename(columns={"key": "year", "count": "n"}).sort_values("year")
    return df


## Q1. How many works have “artificial intelligence” in the **title**?
We use the `title.search` filter and read `meta.count`.


In [ ]:
q1_count = count_title_contains("artificial intelligence")
print({"count_title_contains_AI": q1_count})

## Q2. Since 2020, group by **author affiliation country**
We filter `publication_year:2020-current` and group by `authorships.institutions.country_code`. Then we sort by counts and optionally map codes to names.


In [ ]:
q2 = group_by_country_since_2020("artificial intelligence")
try:
    import pycountry
    def code_to_name(code):
        try:
            return pycountry.countries.get(alpha_2=code).name
        except Exception:
            return code
    if not q2.empty:
        q2["country"] = q2["country_code"].map(code_to_name)
        q2 = q2[["country_code", "country", "n"]]
except Exception:
    pass
q2.head(20)

## Q3. Top 10 most cited AI papers since 2020
We sort by `cited_by_count:desc` and extract title, citations, authors, and institutions.


In [ ]:
q3 = top_cited_ai_since_2020()
q3

## Q4. Year-to-year trend since 2000: 'artificial intelligence' vs 'LLM' vs 'GPT' (add one more)
We use `group_by=publication_year` for each query term on titles and merge for a multi-series plot.


In [ ]:
terms = ["artificial intelligence", "LLM", "GPT", "machine learning"]  # edit the last one to your interest
series = []
for term in terms:
    df = yearly_counts(term, 2000)
    df["term"] = term
    series.append(df)

q4 = pd.concat(series, ignore_index=True)
pivot = q4.pivot(index="year", columns="term", values="n").fillna(0)
pivot.tail()

In [ ]:
# Plot
ax = pivot.plot(figsize=(10,6))
ax.set_title("Works with keyword in TITLE, by publication year")
ax.set_xlabel("Year")
ax.set_ylabel("Count of works")
plt.tight_layout()
plt.show()

## Quality checks & notes
- Confirm sample sizes and missing values in returned frames.
- Because `title.search` is literal substring matching on titles, it may count false positives (e.g., *GPT* could match unrelated abbreviations). Consider adding context terms or using `search=` (which also looks at abstracts) if appropriate.
- For country grouping, counts reflect **any** author affiliation; a multi-country paper is counted for each country represented.


In [ ]:
# Basic QC examples
print("Q2 total rows (countries):", 0 if 'q2' not in globals() or q2 is None else len(q2))
print("Q3 top rows:")
display(q3.head() if 'q3' in globals() else None)
print("Missing checks (Q3):")
display(q3.isna().sum() if 'q3' in globals() else None)


## Methodology write-up (edit these bullets in your own words)
- **API usage**: `/works` endpoint with `filter=title.search:<term>` for title-only search; `publication_year` range for time filters; `group_by` for country and year aggregation; `sort=cited_by_count:desc` for top-cited.
- **Funding note**: We passed `mailto` for polite rate-limiting; consider small pauses or retries when paginating.
- **Limitations**: Title-only search can undercount (papers with synonymous phrasing) or overcount (ambiguous acronyms). Country grouping double-counts a multi-country paper by design.
- **Reproducibility**: All parameters are set in functions; change the terms in one place and re-run.
